In [1]:
import sys
import os
# 1. Nạp thư mục gốc vào Python Path
sys.path.append(os.path.abspath(".."))

import json
import httpx
from app.core.config import settings

# 2. Định nghĩa thông tin URL
LOGIN_URL = f"{settings.NET_BACKEND_URL}/api/TokenAuth/Authenticate"
SEARCH_URL = f"{settings.NET_BACKEND_URL}/api/RequestDoc/TR_REQUEST_DOC_Search"

auth_payload = {
    "userNameOrEmailAddress": "baotq",
    "password": "Gsoft@#hai0hai6"
}

with httpx.Client(timeout=10.0) as client:
    print(f"🔑 1. Đang đăng nhập lấy Token...")
    login_res = client.post(LOGIN_URL, json=auth_payload)
    
    if login_res.status_code != 200:
        print(f"❌ Đăng nhập thất bại: {login_res.text}")
    else:
        login_data = login_res.json()
        token = login_data.get("result", {}).get("accessToken")
        print(f"✅ Đăng nhập thành công!")
        
        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json",
            "Accept": "application/json"
        }
        
        # 3. Payload ĐẦY ĐỦ các trường bắt buộc của Stored Procedure
        search_payload = {
            "maxResultCount": 10,  # Số dòng tối đa
            "skipCount": 0,        # Bỏ qua 0 dòng (trang 1)
            "reQ_CODE": "PUR/2025/000052",      # Tìm theo số tờ trình (để rỗng = tất cả)
            "type": "DVKD",        # Loại nghiệp vụ (DVKD / DVMS / DVDM)
            "tlnamE_USER": "baotq" # Username người tìm kiếm
        }
        
        print(f"\n📡 2. Đang gọi API Tra cứu Tờ trình...")
        search_res = client.post(SEARCH_URL, json=search_payload, headers=headers)
        
        print(f"📊 Status Code: {search_res.status_code}")
        if search_res.status_code == 200:
            data = search_res.json()
            print("\n🎉 KẾT QUẢ DỮ LIỆU TỜ TRÌNH TRẢ VỀ THÀNH CÔNG (JSON):")
            print(json.dumps(data, indent=2, ensure_ascii=False))
        else:
            print(f"❌ Lỗi tra cứu: {search_res.text}")


🔑 1. Đang đăng nhập lấy Token...
✅ Đăng nhập thành công!

📡 2. Đang gọi API Tra cứu Tờ trình...
📊 Status Code: 200

🎉 KẾT QUẢ DỮ LIỆU TỜ TRÌNH TRẢ VỀ THÀNH CÔNG (JSON):
{
  "result": {
    "totalCount": 1,
    "items": [
      {
        "reQ_ID": "TRRD00000269630",
        "reQ_CODE": "PUR/2025/000052",
        "brancH_DVMS": null,
        "brancH_NAME_DVMS": "Hội sở",
        "reQ_NAME": null,
        "typE_JOB": "",
        "useR_JOB": "",
        "useR_JOB_NAME": "",
        "rolE_LOGIN": null,
        "procesS_DESC": null,
        "reQ_DT": "2025-12-22T00:00:00",
        "reQ_TYPE": 1,
        "reF_ID": 137228,
        "reQ_CONTENT": null,
        "reQ_PARENT_CODE": null,
        "contracT_ID": null,
        "suP_ID": null,
        "suP_NAME": null,
        "suP_ADDR": null,
        "totaL_AMT": 50000000.0,
        "notes": null,
        "recorD_STATUS": "1",
        "makeR_ID": "baotq",
        "creatE_DT": "2025-12-22T14:31:29",
        "autH_STATUS": "A",
        "brancH_CREATE_

In [2]:
# Import cả 4 tools
from app.ai.agent.procurement.tools import (
    search_request_docs,
    get_request_doc_detail,
    check_plan_budget_detail,
    get_po_master_status
)

print("=== 1. TEST TOOL TRA CỨU DANH SÁCH TỜ TRÌNH ===")
res1 = await search_request_docs.ainvoke({"so_to_trinh": "PUR/2025/000052"})
print(res1)

print("\n=== 2. TEST TOOL CHI TIẾT TỜ TRÌNH ===")
res2 = await get_request_doc_detail.ainvoke({"req_id": "TRRD00000269696"})
print(res2)

print("\n=== 3. TEST TOOL KẾ HOẠCH MUA SẮM ===")
res3 = await check_plan_budget_detail.ainvoke({"ma_ke_hoach": ""})
print(res3[:400]) # In 400 ký tự đầu

print("\n=== 4. TEST TOOL ĐƠN HÀNG PO ===")
res4 = await get_po_master_status.ainvoke({"ma_po": ""})
print(res4[:400]) # In 400 ký tự đầu

=== 1. TEST TOOL TRA CỨU DANH SÁCH TỜ TRÌNH ===
Tìm thấy tổng cộng 1 tờ trình. Dưới đây là thông tin chi tiết:

1. Số Tờ trình: PUR/2025/000052
   - Mã hệ thống: TRRD00000269630
   - Trạng thái duyệt: Đã duyệt (Chờ đầu mối mua sắm xử lý)
   - Người tạo: Trương Quang Bảo (Phòng Hỗ trợ)
   - Đơn vị: Hội sở
   - Tổng tiền đề xuất: 50,000,000 VNĐ
   - Ngày lập: 2025-12-22T00:00:00
   - Trích yếu / Lý do: chu trương mua sắm
   - Mã Kế hoạch liên kết: 0049/2025/TTr-0690905

=== 2. TEST TOOL CHI TIẾT TỜ TRÌNH ===
Không tìm thấy chi tiết cho Tờ trình có REQ_ID 'TRRD00000269696'.

=== 3. TEST TOOL KẾ HOẠCH MUA SẮM ===
Không tìm thấy Kế hoạch mua sắm nào khớp với từ khóa ''.

=== 4. TEST TOOL ĐƠN HÀNG PO ===
Tìm thấy tổng cộng 4308 Đơn hàng PO. Chi tiết:

1. Mã Đơn hàng PO: PO069/26/0006
   - Tên gói/Nội dung PO: Tờ trình chủ trương
   - Trạng thái duyệt: Đã duyệt
   - Nhà cung cấp: testzzzzz
   - Tổng giá trị PO: 12,765 VNĐ
   - Ngày lập PO: 2026-03-04T00:00:00
   - Hạn giao hàng: 2025-05-30T00

In [3]:
await search_request_docs.ainvoke({"so_to_trinh": "PUR/2026/000088"})

"Không tìm thấy tờ trình nào khớp với mã 'PUR/2026/000088'."